# Notebook 06 — Path-Dependence: What an Arbitrage-Dirty Surface Actually Costs

NB05 priced the audit, but every object it priced is **terminal or static**: held-out quote
repricing (§A), Breeden–Litzenberger densities (§C), the 30-day variance-swap rate (§D). Each
of those depends on the surface through a *marginal* distribution at a single maturity. That is
exactly the place where an arbitrage-dirty surface looks least damaged, because the fitting
objective is itself a marginal-distribution objective: a surface that fits vanillas well
reproduces the marginals well almost by construction.

The Dupire local volatility, however,

$$\sigma^2_{\mathrm{loc}}(k,\tau) \;=\; \frac{\partial_\tau w(k,\tau)}{g(k,\tau)},$$

is not a marginal object — it is the **instantaneous dynamics**. A defect in $g$ between the
collocation nodes does not merely perturb a terminal density; it perturbs the diffusion
coefficient along every path that visits that region. The economic consequence is therefore
concentrated where NB05 does not look: in path-dependent payoffs.

**This notebook closes that gap with one instrument** — a discretely monitored down-and-out
call — under a controlled experiment with an *independent* ground truth.

| § | Object | What it establishes |
|---|--------|---------------------|
| **B1** | Heston truth engine | An independent data-generating model: characteristic-function vanillas (exact) and Monte Carlo paths. Neither surface family is used to define truth, so the comparison is not circular. |
| **B2** | Clean and dirty surfaces | A GJ-certified SSVI prior fitted to Heston-generated quotes (clean), and node-aligned perturbations that are **exactly zero at every quoted strike** (dirty). Vanilla RMSE is matched by construction, so any downstream difference cannot be a fit-quality difference. |
| **B3** | Dupire extraction | Local variance, the repair rate, and a vega-conditioning mask that separates *genuine* arbitrage violations from FD artefacts in the deep wings. |
| **B4** | Error budget | Three separately measured gaps: PDE-vs-MC (numerics), local-vol-vs-Heston (model class), and dirty-vs-clean (the quantity of interest). The third is only meaningful against the first two. |
| **B5** | Barrier pricing | $\Delta V$ in bp of spot, % of premium, and dollars per contract, swept over barrier level × defect amplitude. |
| **B6** | Delta hedging | Both desks sell at the *same* price and hedge on the *same* Heston paths with their own deltas. The **paired** P&L difference isolates the cost of wrong Greeks from discrete-hedging noise. |
| **B7** | Digital | The butterfly defect read as a probability, linking back to NB05 §C. |
| **B8** | Real packs (gated) | The same barrier machinery applied to NB03/NB04 surfaces on real SPX days, with a matched-RMSE gate. |

**The claim under test.** Not "arbitrage violations are a free lunch" — NB05 §F already shows
they are not. The claim is the complementary one:

> A surface can fit the vanilla market *identically* to a certified arbitrage-free surface and
> still inject economically material error into the price and the hedge of a path-dependent
> product. The cost of a collocation blind spot is not foregone alpha; it is model risk.

**Conventions.** $r=q=0$ and $S_0=F=1$ throughout, so $S$ *is* the forward, log-moneyness
$k=\log(K/F)$ is $\log S$, and every price is quoted per unit of notional. This is the same
normalized (undiscounted, forward-measure) convention as NB05's Black machinery, so all
prices are directly comparable across the two notebooks.

## 0. Config, gating and Black machinery

Everything is env-driven (`NB06_*`); the three section gates let the cluster split the run. Black machinery is identical to NB05's normalized convention.

In [ ]:
import os, glob, re, time, zlib
from datetime import date as _date, datetime as _datetime
from pathlib import Path
from math import erf

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.linalg import solve_banded
from scipy.interpolate import RegularGridInterpolator

OUT_DIR      = Path(os.environ.get("THESIS_OUT_DIR", "data/clean"))
REAL_PARQUET = Path(os.environ.get("THESIS_OPT_PARQUET", str(OUT_DIR / "option_prices_clean.parquet")))
NB03_SURF    = OUT_DIR / "nb03_surfaces"
NB04_SURF    = OUT_DIR / "nb04_surfaces"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---- section gates (every block degrades gracefully) ----
RUN_SYNTHETIC = os.environ.get("NB06_SKIP_SYNTHETIC", "0") != "1"
RUN_HEDGE     = os.environ.get("NB06_SKIP_HEDGE", "0") != "1"
RUN_REAL      = os.environ.get("NB06_SKIP_REAL", "0") != "1"

# ---- numerics ----
SEED         = int(os.environ.get("NB06_SEED", "0"))
T_EXP        = float(os.environ.get("NB06_T", "1.0"))        # option maturity (years)
N_MON        = int(os.environ.get("NB06_N_MON", "126"))      # monitoring == hedging dates
PDE_NX       = int(os.environ.get("NB06_PDE_NX", "1601"))
PDE_SUB      = int(os.environ.get("NB06_PDE_SUB", "2"))
MC_PATHS     = int(os.environ.get("NB06_MC_PATHS", "240000"))
MC_SUB       = int(os.environ.get("NB06_MC_SUB", "4"))       # Euler substeps per monitoring step
HEDGE_PATHS  = int(os.environ.get("NB06_HEDGE_PATHS", "60000"))
CHUNK        = int(os.environ.get("NB06_CHUNK", "15000"))    # path-chunk size (bounds memory)

# ---- experiment design ----
K_STRIKE   = float(os.environ.get("NB06_K_STRIKE", "0.0"))   # ATM call, log-moneyness
BARRIERS   = [float(x) for x in os.environ.get("NB06_BARRIERS", "0.95,0.90,0.85,0.80").split(",")]
AMPS       = [float(x) for x in os.environ.get("NB06_AMPS", "0,2e-4,5e-4,1e-3,2e-3,4e-3").split(",")]
DIRTY_MODE = os.environ.get("NB06_DIRTY_MODE", "ripple")     # ripple | bump | both
DK_LATTICE = float(os.environ.get("NB06_DK", "0.04"))        # quoted log-strike spacing
NOTIONAL   = float(os.environ.get("NB06_NOTIONAL", "100.0")) # index multiplier, for the $ column
SPOT_REF   = float(os.environ.get("NB06_SPOT_REF", "4500.0"))# reference index level, $ column only
RMSE_MATCH_TOL_VP = float(os.environ.get("NB06_RMSE_TOL", "0.05"))   # fit-parity gate (vol pts)

In [ ]:
# ---- local-variance repair band (identical to NB05) ----
V_LOC_FLOOR, V_LOC_CAP = 1e-4, 4.0
VEGA_TRUST = float(os.environ.get("NB06_VEGA_TRUST", "1e-3"))

# ---- Heston ground truth (the third party: neither clean nor dirty) ----
HES = dict(v0=float(os.environ.get("NB06_HES_V0", "0.035")),
           kappa=float(os.environ.get("NB06_HES_KAPPA", "1.8")),
           theta=float(os.environ.get("NB06_HES_THETA", "0.045")),
           sigma=float(os.environ.get("NB06_HES_SIGMA", "0.65")),
           rho=float(os.environ.get("NB06_HES_RHO", "-0.72")))
FELLER = 2 * HES["kappa"] * HES["theta"] - HES["sigma"] ** 2

print(f"synthetic={RUN_SYNTHETIC} hedge={RUN_HEDGE} real={RUN_REAL}")
print(f"Heston truth: {HES} | Feller 2*kappa*theta - sigma^2 = {FELLER:+.4f} "
      f"({'satisfied' if FELLER > 0 else 'violated — full truncation Euler handles v=0'})")
print(f"NB03 packs: {NB03_SURF.exists()} | NB04 packs: {NB04_SURF.exists()} | "
      f"quotes: {REAL_PARQUET.exists()}")

In [ ]:
def _Phi(x):
    return 0.5 * (1.0 + np.vectorize(erf)(np.asarray(x, float) / np.sqrt(2.0)))


def _phi(x):
    return np.exp(-0.5 * np.asarray(x, float) ** 2) / np.sqrt(2 * np.pi)


def black_call(k, w):
    w = np.maximum(np.asarray(w, float), 1e-12)
    sw = np.sqrt(w)
    d1 = (-np.asarray(k, float) + w / 2) / sw
    return _Phi(d1) - np.exp(k) * _Phi(d1 - sw)


def black_put(k, w):
    return black_call(k, w) - 1.0 + np.exp(k)


def implied_w(price, k, is_call, lo=1e-10, hi=25.0, iters=90):
    """Vectorized bisection for total variance from a normalized price (monotone in w)."""
    price = np.asarray(price, float); k = np.asarray(k, float)
    lo = np.full_like(price, lo); hi = np.full_like(price, hi)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        pm = np.where(is_call, black_call(k, mid), black_put(k, mid))
        too_low = pm < price
        lo = np.where(too_low, mid, lo)
        hi = np.where(too_low, hi, mid)
    return 0.5 * (lo + hi)


def durrleman_g(k, w, wk, wkk):
    return (1 - k * wk / (2 * w)) ** 2 - (wk ** 2 / 4) * (1 / w + 0.25) + wkk / 2


_k = np.array([-0.3, 0.0, 0.2]); _w = np.array([0.04, 0.02, 0.06])
assert np.max(np.abs(implied_w(black_call(_k, _w), _k, np.array([True] * 3)) - _w)) < 1e-7
print("Black round-trip self-test OK")

## B1. The Heston truth engine — an independent data-generating model

The circularity trap in this experiment is easy to fall into: if the paths that judge the two
desks are generated from the *clean* surface's own local volatility, the clean desk wins by
construction and the experiment proves nothing. The ground truth must therefore be a third
model, related to neither candidate surface.

We use Heston,

$$dS_t = \sqrt{v_t}\,S_t\,dW_t,\qquad
  dv_t = \kappa(\theta - v_t)\,dt + \sigma\sqrt{v_t}\,dB_t,\qquad
  d\langle W,B\rangle_t = \rho\,dt,$$

with $r=q=0$ and $S_0=1$. It serves two roles, and the separation matters:

1. **Vanilla quotes** come from the *characteristic function* (Lewis' formula), so the synthetic
   market is exact to quadrature error and carries no Monte Carlo noise. These are the quotes
   both surfaces are fitted to.
2. **Barrier truth and hedging paths** come from *Monte Carlo* on the same parameters.

The MC engine is validated against the CF pricer on vanillas below. That validation is what
licenses its use on the barrier, where no closed form exists.

Two properties of this choice are worth stating explicitly for the defence. First, Heston is a
genuine martingale model, so its implied surface is arbitrage-free by construction and its
vanilla prices are internally consistent — the synthetic market has no arbitrage to find.
Second, Heston is *not* a local-volatility model: its Dupire projection reproduces every
marginal but not the joint law across monitoring dates. The resulting local-vol-versus-Heston
gap on a path-dependent payoff is a real, documented effect, and §B4 measures it separately so
that it is never confused with the defect we are studying.

In [ ]:
def heston_cf(u, T, v0, kappa, theta, sigma, rho):
    """E[exp(i u X_T)] with X = log(S_T/S_0), r = q = 0.
    Albrecher et al. "little trap" branch: |g2| < 1, so the complex log never wraps."""
    u = np.asarray(u, dtype=complex)
    xi = kappa - rho * sigma * 1j * u
    d = np.sqrt(xi ** 2 + (sigma ** 2) * (1j * u + u ** 2))
    g2 = (xi - d) / (xi + d)
    e = np.exp(-d * T)
    C = (kappa * theta / sigma ** 2) * ((xi - d) * T - 2.0 * np.log((1 - g2 * e) / (1 - g2)))
    D = ((xi - d) / sigma ** 2) * ((1 - e) / (1 - g2 * e))
    return np.exp(C + D * v0)

In [ ]:
def heston_call(k, T, v0, kappa, theta, sigma, rho, n_quad=512):
    """Lewis formula for C/S0 at log-moneyness k, Gauss-Legendre in u.

    The truncation u_max is MATURITY-AWARE: the integrand decays on a scale set by the
    total standard deviation, so a short maturity needs a much wider u-range. Prices are
    clipped into the no-arbitrage band before any IV inversion; without the clip, a 1e-10
    quadrature residue on a 1e-9 deep-wing price inverts to nonsense."""
    k = np.atleast_1d(np.asarray(k, float))
    scale = np.sqrt(max(v0, theta) * max(T, 1e-4))
    u_max = float(np.clip(60.0 / max(scale, 1e-3), 200.0, 6000.0))
    x, wq = np.polynomial.legendre.leggauss(n_quad)
    u = 0.5 * u_max * (x + 1.0)
    wu = 0.5 * u_max * wq
    cf = heston_cf(u - 0.5j, T, v0, kappa, theta, sigma, rho)
    integ = (np.real(np.exp(-1j * np.outer(k, u)) * cf[None, :]) / (u ** 2 + 0.25)[None, :]) @ wu
    c = 1.0 - np.exp(k / 2) / np.pi * integ
    return np.clip(c, np.maximum(1 - np.exp(k), 0.0) + 1e-13, 1.0 - 1e-13)

In [ ]:
def heston_w(k, tau, p=HES):
    """Heston total implied variance at (k, tau); loops over distinct maturities."""
    k = np.atleast_1d(np.asarray(k, float)); tau = np.atleast_1d(np.asarray(tau, float))
    kk, tt = np.broadcast_arrays(k, tau)
    out = np.empty(kk.shape)
    for t in np.unique(tt):
        m = tt == t
        out[m] = implied_w(heston_call(kk[m], float(t), **p), kk[m], np.ones(int(m.sum()), bool))
    return out

In [ ]:
def heston_chunk(T, n_mon, sub, n_paths, rng, p=HES):
    """One antithetic chunk of full-truncation Euler paths.

    Simulates n_mon*sub substeps but RETURNS ONLY the n_mon+1 monitoring snapshots of
    X = log(S/S0): memory is n_paths x (n_mon+1) rather than n_paths x (n_mon*sub+1),
    which is what makes 10^5-path runs feasible on a shared node."""
    assert n_paths % 2 == 0, "antithetic pairing needs an even path count"
    m = n_paths // 2
    dt = T / (n_mon * sub)
    sdt = np.sqrt(dt)
    Xs = np.zeros((n_paths, n_mon + 1))
    X = np.zeros(n_paths)
    v = np.full(n_paths, float(p["v0"]))
    for j in range(n_mon):
        for _ in range(sub):
            z1 = rng.standard_normal(m); z2 = rng.standard_normal(m)
            z1 = np.concatenate([z1, -z1]); z2 = np.concatenate([z2, -z2])
            dW = sdt * z1
            dB = p["rho"] * dW + np.sqrt(1 - p["rho"] ** 2) * sdt * z2
            vp = np.maximum(v, 0.0)                      # full truncation
            sv = np.sqrt(vp)
            X = X - 0.5 * vp * dt + sv * dW
            v = v + p["kappa"] * (p["theta"] - vp) * dt + p["sigma"] * sv * dB
        Xs[:, j + 1] = X
    return Xs

In [ ]:
# ---------------- self-test 1: degenerate Heston must collapse onto Black ----------------
# v0 = theta and (near-)zero vol-of-vol freeze the variance at v0, so the model must return
# Black with w = v0*T. sigma is set to 1e-4 rather than 0: sigma^2 sits in the denominator of
# both C and D, and pushing it below ~1e-5 costs more to catastrophic cancellation than the
# degeneracy is worth (at sigma=1e-6 the error is 1e-5; at 1e-4 it is 5e-10).
_p0 = dict(v0=0.04, kappa=2.0, theta=0.04, sigma=1e-4, rho=0.0)
_ks = np.array([-0.3, -0.1, 0.0, 0.1, 0.25])
_err = np.max(np.abs(heston_call(_ks, 0.5, **_p0) - black_call(_ks, 0.04 * 0.5)))
print(f"[B1 test 1] zero vol-of-vol vs Black: max abs price error = {_err:.2e}")
assert _err < 1e-6

In [ ]:
# ---------------- self-test 2: MC must reproduce the CF on vanillas ----------------
if RUN_SYNTHETIC:
    _t0 = time.time()
    _rng = np.random.default_rng(SEED + 101)
    _ks = np.array([-0.30, -0.15, 0.0, 0.10, 0.20])
    _tot = np.zeros(len(_ks)); _n = 0; _sS = 0.0
    _target = min(MC_PATHS, 200_000)
    while _n < _target:
        _c = min(CHUNK, _target - _n)
        _Xs = heston_chunk(T_EXP, N_MON, MC_SUB, _c, _rng)
        _ST = np.exp(_Xs[:, -1])
        for _i, _k in enumerate(_ks):
            _tot[_i] += np.maximum(_ST - np.exp(_k), 0.0).sum()
        _sS += _ST.sum(); _n += _c
    _mc = _tot / _n
    _cf = heston_call(_ks, T_EXP, **HES)
    _iv_mc = np.sqrt(implied_w(_mc, _ks, np.ones(len(_ks), bool)) / T_EXP)
    _iv_cf = np.sqrt(implied_w(_cf, _ks, np.ones(len(_ks), bool)) / T_EXP)
    print(f"[B1 test 2] MC vs CF on vanillas ({_n:,} paths, {time.time()-_t0:.1f}s): "
          f"IV error (vol pts) = {np.round((_iv_mc - _iv_cf) * 100, 3)}")
    print(f"[B1 test 2] martingale check E[S_T] - 1 = {_sS/_n - 1:+.5f}")
    assert np.max(np.abs(_iv_mc - _iv_cf)) < 0.005, "MC engine disagrees with the CF pricer"
    print("[B1] MC engine validated on vanillas -> licensed for the barrier, where no "
          "closed form exists.")

## B2. The two surfaces — matched on vanillas by construction

**Clean.** An SSVI prior fitted to the synthetic quotes and calibrated *inside* the
Gatheral–Jacquier region, so it carries the same static certificate used throughout NB03/NB04:

$$w_{\mathrm{SSVI}}(k,\theta_\tau) = \tfrac{\theta_\tau}{2}\Bigl(1 + \rho\varphi k +
  \sqrt{(\varphi k + \rho)^2 + (1-\rho^2)}\Bigr),\qquad \varphi = \eta\,\theta_\tau^{-\gamma}.$$

**Dirty.** The clean surface times a multiplicative defect,
$w_{\mathrm{dirty}} = w_{\mathrm{clean}}\,(1 + a\,\psi(k,\tau))$, in one of two geometries:

- **`ripple`** (default) — $\psi = \sin(2\pi k/\Delta k)\,\mathrm{env}(\tau)$ with $\Delta k$ the
  quoted log-strike spacing. Because $\sin(2\pi k_j/\Delta k) = 0$ at every quoted strike $k_j$,
  the defect is **exactly zero on the entire quote lattice**: the vanilla RMSE is unchanged to
  machine precision, for any amplitude. This is the collocation blind spot in its purest form —
  a surface perfectly controlled *at* the nodes and uncontrolled *between* them, which is the
  geometry the deep smoother and the operator actually produce.
- **`bump`** — a localized Gaussian excursion placed between two listed maturities and between
  two strikes, i.e. inside a single wide gap of the collocation grid.

The amplitude $a$ is a **dose**, not a fitted quantity. Sweeping it traces the dose–response
curve that is the point of the whole notebook: vanilla fit quality on one axis (flat by
construction), downstream damage on the other.

**Why this construction is the right control.** In NB04 the dirty surface is the GNO and the
clean one is SSVI or the prior-embedded operator, but those differ in *both* arbitrage
cleanliness and fit quality, so a downstream price difference is confounded. Here the two
surfaces are the same object up to a defect that the vanilla objective provably cannot see.
Any downstream difference is therefore attributable to the defect alone. §B8 then repeats the
measurement on the real NB03/NB04 packs, where the fit-parity gate has to be enforced
numerically rather than by construction.

In [ ]:
def ssvi_w(k, theta, rho, eta, gamma):
    phi = eta * theta ** (-gamma)
    return 0.5 * theta * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + (1 - rho ** 2)))


def gj_ok(rho, eta, gamma, alpha, beta, t_lo, t_hi):
    """Gatheral-Jacquier Thm 4.1/4.2 on the working theta range (NB03/NB05 convention:
    gamma free in (0,1) via the endpoint-monotonicity lemma, not clipped at 1/2)."""
    th_max = alpha * t_hi ** beta; th_min = alpha * max(t_lo, 1e-6) ** beta
    c1 = eta * th_max ** (1 - gamma) * (1 + abs(rho))
    c2 = eta ** 2 * max(th_max ** (1 - 2 * gamma), th_min ** (1 - 2 * gamma)) * (1 + abs(rho))
    return (0 < gamma < 1) and (c1 < 4) and (c2 <= 4)

In [ ]:
def fit_ssvi(kq, tq, wq, wtq):
    taus = np.unique(tq); t_lo_, t_hi_ = float(tq.min()), float(tq.max())
    th_t, th_v = [], []
    for t in taus:
        kk_, ww_ = kq[tq == t], wq[tq == t]
        j = int(np.argmin(np.abs(kk_)))
        if abs(float(kk_[j])) <= 0.20 and float(ww_[j]) > 0:
            th_t.append(float(t)); th_v.append(float(ww_[j]))
    if len(th_t) >= 2:
        A = np.vstack([np.ones(len(th_t)), np.log(th_t)]).T
        coef, *_ = np.linalg.lstsq(A, np.log(th_v), rcond=None)
        a0, b0 = float(np.exp(coef[0])), float(np.clip(coef[1], 0.3, 1.5))
    else:
        a0, b0 = float(np.median(wq / tq ** 0.95)), 0.95

    def sse(p):
        rho, eta, gamma, alpha, beta = p
        rho = float(np.clip(rho, -0.95, 0.95)); eta = max(eta, 1e-4)
        gamma = float(np.clip(gamma, 0.05, 0.95)); alpha = max(alpha, 1e-6)
        beta = float(np.clip(beta, 0.3, 1.5))
        tot = sum(float(np.sum(wtq[tq == t] * (ssvi_w(kq[tq == t], alpha * t ** beta,
                                                      rho, eta, gamma) - wq[tq == t]) ** 2))
                  for t in taus)
        if not gj_ok(rho, eta, gamma, alpha, beta, t_lo_, t_hi_):
            tot += 1e3
        return tot

    best = None
    for x0 in [(-0.5, 1.0, 0.3, a0, b0), (-0.7, 0.6, 0.4, a0, b0)]:
        r = minimize(sse, x0, method="Nelder-Mead", options={"maxiter": 2000})
        if best is None or r.fun < best.fun:
            best = r
    rho, eta, gamma, alpha, beta = best.x
    p = dict(rho=float(np.clip(rho, -0.95, 0.95)), eta=float(max(eta, 1e-4)),
             gamma=float(np.clip(gamma, 0.05, 0.95)), alpha=float(max(alpha, 1e-6)),
             beta=float(np.clip(beta, 0.3, 1.5)))
    assert gj_ok(**p, t_lo=t_lo_, t_hi=t_hi_), "SSVI escaped the GJ region"
    return p

In [ ]:
if RUN_SYNTHETIC:
    # ---- the listed lattice: UNIFORM log-strike spacing DK_LATTICE at every maturity.
    # Real SPX strikes are uniform in K; over the liquid near-ATM band that is very nearly
    # uniform in k. Uniformity is what lets the `ripple` defect vanish on the whole lattice.
    QUOTE_TAUS = np.array([float(x) for x in
                           os.environ.get("NB06_QUOTE_TAUS",
                                          "0.08,0.17,0.25,0.50,0.75,1.00,1.50").split(",")])
    _kq, _tq = [], []
    for _t in QUOTE_TAUS:
        _s = np.sqrt(HES["theta"] * _t)                      # ~1 std of total move
        _lo = -int(np.floor(3.0 * _s / DK_LATTICE))
        _hi = int(np.floor(2.0 * _s / DK_LATTICE))
        _kk = DK_LATTICE * np.arange(_lo, _hi + 1)
        _kq.append(_kk); _tq.append(np.full(len(_kk), _t))
    KQ = np.concatenate(_kq); TQ = np.concatenate(_tq)
    WQ = heston_w(KQ, TQ)
    IVQ = np.sqrt(WQ / TQ)
    print(f"[B2] synthetic market: {len(KQ)} quotes, {len(QUOTE_TAUS)} maturities, "
          f"k in [{KQ.min():+.3f},{KQ.max():+.3f}], lattice spacing dk = {DK_LATTICE}")

    _wt = 1.0 / (4 * np.maximum(WQ, 1e-10) * TQ); _wt = _wt / _wt.mean()
    SSVI_P = fit_ssvi(KQ, TQ, WQ, _wt)
    print(f"[B2] SSVI prior: { {k: round(v, 4) for k, v in SSVI_P.items()} }  "
          f"GJ-certified = {gj_ok(**SSVI_P, t_lo=TQ.min(), t_hi=TQ.max())}")

In [ ]:
if RUN_SYNTHETIC:
    def w_clean(k, tau):
        return ssvi_w(k, SSVI_P["alpha"] * tau ** SSVI_P["beta"],
                      SSVI_P["rho"], SSVI_P["eta"], SSVI_P["gamma"])

    # ---- defect geometries -------------------------------------------------------------
    TAU_ENV_C = float(np.exp(np.mean(np.log(QUOTE_TAUS))))    # geometric centre of the term
    TAU_ENV_W = float(os.environ.get("NB06_TAU_ENV_W", "0.9"))
    BUMP_KC = float(os.environ.get("NB06_BUMP_KC", "-0.11"))
    BUMP_TC = float(os.environ.get("NB06_BUMP_TC", str(np.sqrt(0.50 * 0.75))))
    BUMP_SK = float(os.environ.get("NB06_BUMP_SK", "0.030"))
    BUMP_ST = float(os.environ.get("NB06_BUMP_ST", "0.16"))

    def _tau_env(tau):
        return np.exp(-0.5 * ((np.log(np.maximum(tau, 1e-8)) - np.log(TAU_ENV_C)) / TAU_ENV_W) ** 2)

    def psi_ripple(k, tau):
        """Zero at every quoted strike by construction: sin(2*pi*k_j/dk) = 0 for k_j on lattice."""
        return np.sin(2 * np.pi * k / DK_LATTICE) * _tau_env(tau)

    def psi_bump(k, tau):
        """A single Gaussian excursion inside one wide gap of the collocation grid."""
        return (np.exp(-0.5 * ((k - BUMP_KC) / BUMP_SK) ** 2) *
                np.exp(-0.5 * ((np.log(np.maximum(tau, 1e-8)) - np.log(BUMP_TC)) / BUMP_ST) ** 2))

    PSI = dict(ripple=psi_ripple, bump=psi_bump)
    MODES = ["ripple", "bump"] if DIRTY_MODE == "both" else [DIRTY_MODE]

In [ ]:
if RUN_SYNTHETIC:
    # The two geometries live on different amplitude scales and must not share a grid.
    # Curvature is what breaks g, and for the ripple w_kk ~ a*w*(2*pi/dk)^2 -- with dk = 0.04
    # that prefactor is ~2.5e4, so a ripple needs a ~ 1e-3 where a bump of width ~0.03 needs
    # a ~ 5e-2 to inflict comparable damage. Using one grid for both would make the bump look
    # harmless for a reason that has nothing to do with its geometry.
    AMPS_BY_MODE = dict(
        ripple=AMPS,
        bump=[float(x) for x in os.environ.get("NB06_AMPS_BUMP",
                                               "0,0.02,0.04,0.06,0.08").split(",")])
    for _m in MODES:
        print(f"[B2] mode={_m}: amplitude grid {AMPS_BY_MODE[_m]}")

    def make_w_fn(mode, a):
        psi = PSI[mode]
        return lambda k, tau: w_clean(k, tau) * (1 + a * psi(k, tau))

    def vanilla_rmse_vp(fn):
        """Fit quality on the quoted lattice, in vol points — the fit-parity metric."""
        return float(np.sqrt(np.mean((np.sqrt(fn(KQ, TQ) / TQ) - IVQ) ** 2)) * 100)

    RMSE_CLEAN = vanilla_rmse_vp(w_clean)
    print(f"[B2] clean SSVI vanilla RMSE vs the Heston market = {RMSE_CLEAN:.5f} vol pts")
    for _m in MODES:
        _chk = [vanilla_rmse_vp(make_w_fn(_m, a)) for a in AMPS_BY_MODE[_m]]
        print(f"[B2] mode={_m:6s} vanilla RMSE across the amplitude sweep: "
              f"{np.round(_chk, 5)}  (spread {max(_chk)-min(_chk):.2e} vp)")
        if _m == "ripple":
            assert max(_chk) - min(_chk) < 1e-6, \
                "ripple defect leaked into the quote lattice — check DK_LATTICE alignment"
            print("       -> node-aligned: the vanilla objective is BIT-IDENTICAL across doses.")

## B3. Dupire extraction, the repair rate, and one numerical lesson

Local variance is the audit quantities in the denominator and numerator of a single ratio,

$$\sigma^2_{\mathrm{loc}} = \frac{\partial_\tau w}{g},$$

so a calendar violation ($\partial_\tau w < 0$) or a butterfly violation ($g \le 0$) makes it
negative or unbounded. Such a cell cannot be simulated: it must be **repaired** — floored,
capped, or interpolated over — before a single path is drawn. The share of grid cells requiring
repair is reported for every surface, and the repair is the model risk made explicit.

**A numerical lesson worth a paragraph in the thesis.** Building this notebook surfaced a trap.
Computing $g$ by finite differences on an implied-variance grid extracted from *exact* Heston
prices produced $\min g = -0.68$ and a 4.9% repair rate — on a surface that is arbitrage-free by
construction. The violations were entirely an artefact: in the deep wings at short maturity the
Black vega collapses, the price-to-implied-vol inversion becomes ill-conditioned, and the second
difference in $k$ amplifies the residual noise without bound. Restricting attention to cells
where the vega exceeds a threshold, $\varphi(d_1) > 10^{-3}$, the same surface reads
$\min g = +0.35$ and a repair rate of exactly zero.

We therefore audit inside a **vega-trusted core** and extend the local variance flat in $k$
outside it. The point generalizes beyond this notebook: any reported violation rate on a
finite-difference grid is a joint statement about the surface *and* the conditioning of the
extraction, and the two must be separated before a violation is attributed to the model.

In [ ]:
def wing_extend(V, trust):
    """Flat-in-k extension of a field outside the vega-trusted core of each maturity row."""
    out = V.copy()
    for i in range(V.shape[0]):
        idx = np.where(trust[i])[0]
        if len(idx) < 5:
            continue
        out[i, :idx[0]] = V[i, idx[0]]
        out[i, idx[-1] + 1:] = V[i, idx[-1]]
    return out

In [ ]:
def dupire_fields(kg, tg, W, use_trust=True):
    """Returns (vloc_fn, v_raw, invalid_all, invalid_core, G, Wt, trust).

    `invalid_all`  : repair rate over the whole export grid (comparable to NB05 sec. A).
    `invalid_core` : repair rate restricted to the vega-trusted core — the number that
                     actually attributes a violation to the SURFACE rather than to the
                     conditioning of the finite-difference extraction."""
    KK, _ = np.meshgrid(kg, tg)
    Wk = np.gradient(W, kg, axis=1)
    Wkk = np.gradient(Wk, kg, axis=1)
    Wt = np.gradient(W, tg, axis=0)
    G = durrleman_g(kg[None, :], np.maximum(W, 1e-12), Wk, Wkk)

    v = Wt / np.where(np.abs(G) < 1e-12, np.nan, G)
    invalid = ~np.isfinite(v) | (v <= 0) | (G <= 0) | (Wt < 0)

    d1 = (-KK + W / 2) / np.sqrt(np.maximum(W, 1e-12))
    trust = _phi(d1) > VEGA_TRUST if use_trust else np.ones_like(W, bool)
    if use_trust:
        v = wing_extend(np.where(invalid, np.nan, v), trust)
    invalid_core = invalid & trust

    v_rep = np.clip(np.nan_to_num(v, nan=V_LOC_FLOOR), V_LOC_FLOOR, V_LOC_CAP)
    itp = RegularGridInterpolator((tg, kg), v_rep, bounds_error=False, fill_value=None)

    def vloc_fn(x, t):
        x = np.asarray(x, float)
        return itp(np.stack([np.full_like(x, np.clip(t, tg[0], tg[-1])),
                             np.clip(x, kg[0], kg[-1])], -1))

    return vloc_fn, v, invalid, invalid_core, G, Wt, trust

In [ ]:
if RUN_SYNTHETIC:
    # export grid: fine in k (the ripple must be resolved), hybrid in tau (NB03 convention)
    NK_EXP = int(os.environ.get("NB06_NK", "1201"))
    NT_EXP = int(os.environ.get("NB06_NT", "81"))
    T_LO, T_HI = 0.015, max(1.6, T_EXP * 1.5)
    KG = np.linspace(-0.75, 0.45, NK_EXP)
    TG = np.unique(np.round(np.concatenate([
        np.exp(np.linspace(np.log(T_LO), np.log(T_HI), NT_EXP)),
        np.linspace(T_LO, T_HI, NT_EXP)]), 10))
    _pts_per_wave = DK_LATTICE / (KG[1] - KG[0])
    print(f"[B3] export grid {len(KG)} x {len(TG)}; {_pts_per_wave:.0f} k-nodes per ripple "
          f"wavelength (need >> 2 to resolve w_kk)")
    assert _pts_per_wave > 8, "k-grid too coarse to resolve the defect's curvature"

    # ---- the reference Heston surface: the numerical lesson, demonstrated ----
    _W_ref = np.empty((len(TG), len(KG)))
    _t0 = time.time()
    for _i, _t in enumerate(TG):
        _W_ref[_i] = implied_w(heston_call(KG, float(_t), **HES), KG, np.ones(len(KG), bool))
    VLOC_REF, _vr, _inv_all, _inv_core, _G_ref, _Wt_ref, _trust = dupire_fields(KG, TG, _W_ref)
    print(f"[B3] reference Heston surface built in {time.time()-_t0:.1f}s")
    print(f"     WITHOUT vega mask: min g = {_G_ref.min():+.4f}, repair = {100*_inv_all.mean():.2f}%")
    print(f"     WITHIN vega-trusted core ({100*_trust.mean():.0f}% of cells): "
          f"min g = {_G_ref[_trust].min():+.4f}, repair = {100*_inv_core.mean():.4f}%")
    print("     -> the violations were an extraction artefact, not a property of Heston.")

In [ ]:
if RUN_SYNTHETIC:
    # ---- clean and dirty surfaces on the same grid ----
    _KK, _TT = np.meshgrid(KG, TG)
    SURF = {}
    for _m in MODES:
        for _a in AMPS_BY_MODE[_m]:
            _fn = make_w_fn(_m, _a)
            _W = _fn(_KK, _TT)
            _vfn, _v, _ia, _ic, _G, _Wt, _tr = dupire_fields(KG, TG, _W)
            _name = "clean" if _a == 0 else f"{_m}_a{_a:g}"
            if _name in SURF:
                continue
            SURF[_name] = dict(mode=_m, a=_a, w_fn=_fn, vloc=_vfn, W=_W, G=_G, Wt=_Wt,
                               repair_all=100 * float(_ia.mean()),
                               repair_core=100 * float(_ic.mean()),
                               min_g=float(_G[_tr].min()), min_wt=float(_Wt[_tr].min()),
                               rmse_vp=vanilla_rmse_vp(_fn))
    rows_B3 = [dict(surface=n, mode=d["mode"], amplitude=d["a"], vanilla_rmse_vp=d["rmse_vp"],
                    min_g=d["min_g"], min_wt=d["min_wt"],
                    repair_core_pct=d["repair_core"], repair_all_pct=d["repair_all"])
               for n, d in SURF.items()]
    dfB3 = pl.DataFrame(rows_B3).sort(["mode", "amplitude"])
    dfB3.write_parquet(OUT_DIR / "nb06_surface_audit.parquet")
    print("\n[B3] surface audit (note the vanilla RMSE column against the repair column):")
    print(dfB3)

### B3b. Figure — the defect the vanilla fit cannot see

In [ ]:
if RUN_SYNTHETIC and len(SURF) > 1:
    _worst = max(SURF.items(), key=lambda kv: kv[1]["repair_core"])[0]
    _pair = ["clean", _worst]
    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.13, horizontal_spacing=0.09,
                        subplot_titles=[f"{n}: sigma_loc (red = repaired)" for n in _pair]
                                       + [f"{n}: Durrleman g (red = g <= 0)" for n in _pair])
    for _j, _n in enumerate(_pair, start=1):
        _d = SURF[_n]
        _vfn, _v, _ia, _ic, _G, _Wt, _tr = dupire_fields(KG, TG, _d["W"])
        _vr = np.clip(np.nan_to_num(_v, nan=V_LOC_FLOOR), V_LOC_FLOOR, V_LOC_CAP)
        fig.add_trace(go.Heatmap(z=np.sqrt(_vr), x=KG, y=TG, colorscale="Viridis",
                                 zmin=0.0, zmax=0.6, showscale=(_j == 2),
                                 colorbar=dict(title="sigma_loc", len=0.42, y=0.79)), 1, _j)
        fig.add_trace(go.Heatmap(z=np.where(_ic, 1.0, np.nan), x=KG, y=TG, showscale=False,
                                 colorscale=[[0, "rgba(214,39,40,0.9)"],
                                             [1, "rgba(214,39,40,0.9)"]]), 1, _j)
        fig.add_trace(go.Heatmap(z=np.where(_tr, _G, np.nan), x=KG, y=TG, colorscale="RdBu",
                                 zmid=0.0, showscale=(_j == 2),
                                 colorbar=dict(title="g", len=0.42, y=0.22)), 2, _j)
    for _r in (1, 2):
        for _c in (1, 2):
            fig.update_xaxes(title_text="log-moneyness k", row=_r, col=_c)
            fig.update_yaxes(title_text="tau", row=_r, col=_c)
    fig.update_layout(width=1080, height=760,
                      title=f"B3 — clean vs {_worst}: identical vanilla fit "
                            f"({SURF['clean']['rmse_vp']:.5f} vs {SURF[_worst]['rmse_vp']:.5f} vp), "
                            f"repair {SURF['clean']['repair_core']:.2f}% vs "
                            f"{SURF[_worst]['repair_core']:.2f}%")
    fig.show()

    # the smile slice: what a quote-level eye would see
    _tau_show = float(np.exp(np.mean(np.log(QUOTE_TAUS))))
    fig2 = go.Figure()
    _kk = np.linspace(KQ.min(), KQ.max(), 600)
    for _n in _pair:
        fig2.add_trace(go.Scatter(x=_kk, y=np.sqrt(SURF[_n]["w_fn"](_kk, _tau_show) / _tau_show) * 100,
                                  mode="lines", name=_n))
    _lat = KQ[np.isclose(TQ, QUOTE_TAUS[np.argmin(np.abs(QUOTE_TAUS - _tau_show))])]
    fig2.add_trace(go.Scatter(x=_lat, y=np.sqrt(w_clean(_lat, _tau_show) / _tau_show) * 100,
                              mode="markers", name="quoted strikes",
                              marker=dict(size=9, symbol="x", color="black")))
    fig2.update_layout(width=920, height=420,
                       title=f"B3 — the two smiles at tau = {_tau_show:.2f}: the defect lives "
                             f"strictly BETWEEN the quoted strikes",
                       xaxis_title="log-moneyness k", yaxis_title="implied vol (%)")
    fig2.show()

## B4. The error budget — three gaps, measured separately

A single number ("the dirty surface misprices the barrier by $x$ bp") is worthless unless the
reader knows what else could produce $x$ bp. Three distinct gaps sit between the Heston truth
and a barrier price computed from a fitted surface, and they are measured independently here:

1. **Numerical gap.** PDE versus Monte Carlo *under the same local volatility*. This is pure
   discretization and must be within Monte Carlo noise. It validates the pricer.
2. **Model-class gap.** Local volatility versus Heston, both reproducing the same vanilla
   marginals. Dupire matches every marginal but not the joint law across monitoring dates, so a
   path-dependent payoff separates them. This gap is real, irreducible, and has nothing to do
   with arbitrage violations — it is the benchmark scale against which the third gap is judged.
3. **Defect gap.** Dirty versus clean, both fitted to the same quotes with the same RMSE. This
   is the quantity of interest.

Two further validations are printed: the **Dupire round trip** (a vanilla repriced through the
extracted local volatility must return the surface's own price — this is the strongest single
check that the extraction and the PDE are consistent), and the PDE's convergence under grid
refinement.

**One implementation note that dominates barrier accuracy.** The spatial grid is constructed so
that the barrier falls *exactly on a node*. An unaligned barrier smears the absorbing condition
over one cell and produces a first-order error in the price, which is easily larger than the
effect being measured. The strike is treated the same way. Monitoring is discrete and the PDE
applies the knock-out only on monitoring dates, matching the Monte Carlo convention exactly;
under constant volatility this reproduces the Broadie–Glasserman–Kou continuity-corrected
closed form, which is the self-test below.

In [ ]:
def pde_barrier(vloc_fn, T, k_strike, k_barrier=None, is_call=True,
                n_mon=None, nx=None, sub=None, x_pad=6.0, ref_vol=0.25):
    """Backward Crank-Nicolson for   dV/dt + 0.5 v(x,t) (V_xx - V_x) = 0,  r = q = 0,
    x = log(S/S0). `k_barrier` = log(B/S0); None = vanilla.

    The grid is ALIGNED so the barrier sits exactly on a node -- the single biggest
    accuracy lever for barrier PDEs. The domain deliberately extends BELOW the barrier
    because monitoring is discrete: a path may dip and recover between monitoring dates,
    and the knock-out is applied only on those dates (same convention as the MC).

    Returns (t_nodes, x, V) with V.shape = (n_mon+1, nx). V[j] is the value function at
    monitoring date j -- exactly what a hedging desk interpolates in B6."""
    n_mon = N_MON if n_mon is None else n_mon
    nx = PDE_NX if nx is None else nx
    sub = PDE_SUB if sub is None else sub

    half = x_pad * ref_vol * np.sqrt(max(T, 1e-6))
    x_lo = min(-half, (k_barrier - half / 2) if k_barrier is not None else -half)
    x_hi = max(half, k_strike + half / 2)
    if k_barrier is not None:
        n_lo = max(int(round((k_barrier - x_lo) / ((x_hi - x_lo) / (nx - 1)))), 8)
        dx = (k_barrier - x_lo) / n_lo
        n_hi = int(np.ceil((x_hi - k_barrier) / dx))
        x = k_barrier + dx * np.arange(-n_lo, n_hi + 1)
    else:
        x = np.linspace(x_lo, x_hi, nx)
        dx = x[1] - x[0]

    V = (np.maximum(np.exp(x) - np.exp(k_strike), 0.0) if is_call
         else np.maximum(np.exp(k_strike) - np.exp(x), 0.0))
    if k_barrier is not None:
        V[x <= k_barrier + 1e-12] = 0.0

    t_nodes = np.linspace(0.0, T, n_mon + 1)
    out = np.zeros((n_mon + 1, len(x)))
    out[-1] = V
    dt_sub = (T / n_mon) / sub
    v_top = (np.exp(x[-1]) - np.exp(k_strike)) if is_call else 0.0

    for j in range(n_mon - 1, -1, -1):
        for s in range(sub):
            t_mid = t_nodes[j] + (s + 0.5) * dt_sub
            v = np.clip(vloc_fn(x, t_mid), 1e-6, V_LOC_CAP)
            a = 0.5 * v / dx ** 2
            b = -0.5 * v / (2 * dx)
            L_lo, L_di, L_up = a - b, -2.0 * a, a + b
            th = 0.5                                   # Crank-Nicolson
            rhs = V.copy()
            rhs[1:-1] = (V[1:-1] + (1 - th) * dt_sub *
                         (L_lo[1:-1] * V[:-2] + L_di[1:-1] * V[1:-1] + L_up[1:-1] * V[2:]))
            ab = np.zeros((3, len(x)))
            ab[0, 1:] = -th * dt_sub * L_up[:-1]
            ab[1, :] = 1.0 - th * dt_sub * L_di
            ab[2, :-1] = -th * dt_sub * L_lo[1:]
            ab[1, 0] = 1.0; ab[0, 1] = 0.0             # Dirichlet: worthless at the floor
            ab[1, -1] = 1.0; ab[2, -2] = 0.0           # Dirichlet: intrinsic at the ceiling
            rhs[0] = 0.0
            rhs[-1] = v_top
            V = solve_banded((1, 1), ab, rhs)
        if k_barrier is not None:
            V[x <= k_barrier + 1e-12] = 0.0            # monitoring date
        out[j] = V
    return t_nodes, x, out

In [ ]:
def price_delta(vloc_fn, T, k_strike, k_barrier, **kw):
    """Price and spot-delta at S = 1. delta = dV/dS = e^{-x} dV/dx."""
    _, x, V = pde_barrier(vloc_fn, T, k_strike, k_barrier, True, **kw)
    D = np.gradient(V, x, axis=1) * np.exp(-x)[None, :]
    return float(np.interp(0.0, x, V[0])), float(np.interp(0.0, x, D[0])), x, V, D

In [ ]:
def lv_barrier_mc(vloc_fn, T, k_strike, k_barrier, n_mon, n_paths, seed=0, sub=4, chunk=None):
    """Local-vol MC with the SAME discrete monitoring as the PDE. Used only to separate
    PDE numerics from the local-vol/stochastic-vol model gap."""
    chunk = CHUNK if chunk is None else chunk
    rng = np.random.default_rng(seed)
    tot = tot2 = 0.0; n = 0
    dt = (T / n_mon) / sub
    while n < n_paths:
        m = max(min(chunk, n_paths - n) // 2, 1)
        X = np.zeros(2 * m); alive = np.ones(2 * m, bool)
        for j in range(n_mon):
            for s in range(sub):
                v = np.clip(vloc_fn(X, (j * sub + s) * dt), 1e-6, V_LOC_CAP)
                z = rng.standard_normal(m); z = np.concatenate([z, -z])
                X = X - 0.5 * v * dt + np.sqrt(v * dt) * z
            alive &= X > k_barrier
        pay = np.where(alive, np.maximum(np.exp(X) - np.exp(k_strike), 0.0), 0.0)
        tot += pay.sum(); tot2 += (pay ** 2).sum(); n += 2 * m
    mu = tot / n
    return mu, float(np.sqrt(max(tot2 / n - mu ** 2, 0) / n)), n

In [ ]:
def heston_barrier_mc(T, k_strike, k_barrier, n_mon, n_paths, seed=0, sub=None, chunk=None):
    """Heston TRUTH for the same discretely monitored down-and-out call."""
    sub = MC_SUB if sub is None else sub
    chunk = CHUNK if chunk is None else chunk
    rng = np.random.default_rng(seed)
    tot = tot2 = 0.0; n = 0
    while n < n_paths:
        c = max(min(chunk, n_paths - n) // 2 * 2, 2)
        Xs = heston_chunk(T, n_mon, sub, c, rng)
        alive = (Xs[:, 1:] > k_barrier).all(axis=1)
        pay = np.where(alive, np.maximum(np.exp(Xs[:, -1]) - np.exp(k_strike), 0.0), 0.0)
        tot += pay.sum(); tot2 += (pay ** 2).sum(); n += c
    mu = tot / n
    return mu, float(np.sqrt(max(tot2 / n - mu ** 2, 0) / n)), n

In [ ]:
# ---------------- self-test: constant vol vs the BGK continuity-corrected closed form ----
def do_call_closed(S, K, B, sig, T):
    """Reiner-Rubinstein down-and-out call, r = q = 0, CONTINUOUS monitoring, B < K."""
    lam, sT = 0.5, sig * np.sqrt(T)
    x1 = np.log(S / K) / sT + lam * sT
    y1 = np.log(B ** 2 / (S * K)) / sT + lam * sT
    c = S * _Phi(x1) - K * _Phi(x1 - sT)
    corr = S * (B / S) ** (2 * lam) * _Phi(y1) - K * (B / S) ** (2 * lam - 2) * _Phi(y1 - sT)
    return float(c - corr)


_sig = 0.25
_cv = lambda x, t: np.full_like(np.asarray(x, float), _sig ** 2)
_pv, _, _, _, _ = price_delta(_cv, T_EXP, 0.0, None)
print(f"[B4 test] vanilla PDE = {_pv:.6f} vs Black = {float(black_call(0.0, _sig**2*T_EXP)):.6f} "
      f"(diff {1e4*(_pv - float(black_call(0.0, _sig**2*T_EXP))):+.3f} bp)")
assert abs(_pv - float(black_call(0.0, _sig ** 2 * T_EXP))) < 5e-4

for _B in [0.90, 0.85]:
    _p, _, _, _, _ = price_delta(_cv, T_EXP, 0.0, np.log(_B))
    _Beff = _B * np.exp(-0.5826 * _sig * np.sqrt(T_EXP / N_MON))   # Broadie-Glasserman-Kou
    _ref = do_call_closed(1.0, 1.0, _Beff, _sig, T_EXP)
    print(f"[B4 test] B={_B}: PDE (discrete, {N_MON} dates) = {_p:.6f} vs "
          f"BGK-shifted closed form = {_ref:.6f} (diff {1e4*(_p-_ref):+.2f} bp of spot)")

In [ ]:
if RUN_SYNTHETIC:
    rows_B4 = []
    for Bmn in BARRIERS:
        kB = np.log(Bmn)
        # 1. numerics: PDE vs MC under the SAME (clean) local vol
        p_pde, d_pde, _, _, _ = price_delta(SURF["clean"]["vloc"], T_EXP, K_STRIKE, kB)
        p_lvmc, se_lv, n_lv = lv_barrier_mc(SURF["clean"]["vloc"], T_EXP, K_STRIKE, kB,
                                            N_MON, min(MC_PATHS, 120_000), seed=SEED + 5)
        # 2. model class: local vol vs Heston
        p_hes, se_h, n_h = heston_barrier_mc(T_EXP, K_STRIKE, kB, N_MON, MC_PATHS, seed=SEED + 11)
        # 3. Dupire round trip on the vanilla (no barrier)
        p_van_pde, _, _, _, _ = price_delta(SURF["clean"]["vloc"], T_EXP, K_STRIKE, None)
        p_van_surf = float(black_call(K_STRIKE, SURF["clean"]["w_fn"](K_STRIKE, T_EXP)))
        rows_B4.append(dict(
            barrier=Bmn, pde=p_pde, lv_mc=p_lvmc, lv_mc_se=se_lv,
            heston_mc=p_hes, heston_mc_se=se_h,
            numerics_gap_bp=1e4 * (p_pde - p_lvmc),
            numerics_gap_in_se=(p_pde - p_lvmc) / max(se_lv, 1e-12),
            model_class_gap_bp=1e4 * (p_lvmc - p_hes),
            model_class_gap_pct=100 * (p_lvmc / p_hes - 1),
            dupire_roundtrip_bp=1e4 * (p_van_pde - p_van_surf)))
    dfB4 = pl.DataFrame(rows_B4)
    dfB4.write_parquet(OUT_DIR / "nb06_error_budget.parquet")
    print("[B4] error budget (bp = basis points OF SPOT, spot = 1):")
    print(dfB4.select(["barrier", "pde", "lv_mc", "heston_mc", "numerics_gap_bp",
                       "numerics_gap_in_se", "model_class_gap_bp", "model_class_gap_pct",
                       "dupire_roundtrip_bp"]))
    print("\n[B4] Reading: `numerics_gap_in_se` should sit within ~2 MC standard errors "
          "(the PDE is validated). `model_class_gap_pct` is the irreducible local-vol vs "
          "stochastic-vol difference -- the scale any defect effect must be judged against. "
          "`dupire_roundtrip_bp` near zero certifies that the local-vol extraction and the "
          "PDE are mutually consistent on the vanilla the surface was fitted to.")

## B5. Barrier pricing — the dose–response

The experiment is now a single sweep: barrier level $B/S_0$ × defect amplitude $a$, with the
vanilla RMSE carried alongside every row. Three price units are reported, because a thesis
committee and a desk read different ones:

- **bp of spot** — comparable across barriers and independent of the premium's size;
- **% of the clean premium** — the risk-relative number;
- **dollars per contract** — $\Delta V \times S_{\mathrm{ref}} \times$ multiplier, the number a
  trading desk would actually quote.

The headline figure plots vanilla RMSE on the horizontal axis against barrier mispricing on the
vertical. The clean and dirty surfaces occupy the *same vertical line* — identical fit — while
spreading vertically without bound. That single picture is the notebook's argument: the vanilla
market cannot rank these surfaces, and the exotic desk cannot afford not to.

In [ ]:
if RUN_SYNTHETIC:
    rows_B5 = []
    truth_cache = {}
    _t0 = time.time()
    for Bmn in BARRIERS:
        kB = np.log(Bmn)
        if Bmn not in truth_cache:
            truth_cache[Bmn] = heston_barrier_mc(T_EXP, K_STRIKE, kB, N_MON,
                                                 MC_PATHS, seed=SEED + 11)
        V_truth, se_truth, _ = truth_cache[Bmn]
        V_clean, D_clean, _, _, _ = price_delta(SURF["clean"]["vloc"], T_EXP, K_STRIKE, kB)
        for name, d in SURF.items():
            V, D, _, _, _ = price_delta(d["vloc"], T_EXP, K_STRIKE, kB)
            rows_B5.append(dict(
                surface=name, mode=d["mode"], amplitude=d["a"], barrier=Bmn,
                vanilla_rmse_vp=d["rmse_vp"], repair_core_pct=d["repair_core"],
                min_g=d["min_g"], price=V, delta=D,
                dV_vs_clean_bp=1e4 * (V - V_clean),
                dV_vs_clean_pct=100 * (V / V_clean - 1),
                dV_vs_clean_usd=(V - V_clean) * SPOT_REF * NOTIONAL,
                ddelta_vs_clean=D - D_clean,
                err_vs_truth_bp=1e4 * (V - V_truth),
                err_vs_truth_pct=100 * (V / V_truth - 1),
                truth=V_truth, truth_se_bp=1e4 * se_truth))
    dfB5 = pl.DataFrame(rows_B5).sort(["mode", "barrier", "amplitude"])
    dfB5.write_parquet(OUT_DIR / "nb06_barrier_sweep.parquet")
    print(f"[B5] sweep done in {time.time()-_t0:.0f}s "
          f"({len(BARRIERS)} barriers x {len(SURF)} surfaces)")
    for _m in MODES:
        print(f"\n[B5] mode = {_m}: price impact vs the clean surface (bp of spot)")
        print(dfB5.filter((pl.col("mode") == _m) | (pl.col("amplitude") == 0))
                  .unique(subset=["amplitude", "barrier"])
                  .pivot(values="dV_vs_clean_bp", index="amplitude", on="barrier")
                  .sort("amplitude"))
        print(f"[B5] mode = {_m}: same, as % of the clean premium")
        print(dfB5.filter((pl.col("mode") == _m) | (pl.col("amplitude") == 0))
                  .unique(subset=["amplitude", "barrier"])
                  .pivot(values="dV_vs_clean_pct", index="amplitude", on="barrier")
                  .sort("amplitude"))
    print("\n[B5] vanilla RMSE by amplitude (the column that stays flat):")
    print(dfB5.group_by(["mode", "amplitude"]).agg(
        pl.col("vanilla_rmse_vp").first(), pl.col("repair_core_pct").first(),
        pl.col("min_g").first()).sort(["mode", "amplitude"]))

In [ ]:
if RUN_SYNTHETIC and len(SURF) > 1:
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                        subplot_titles=["Vanilla fit cannot separate these surfaces",
                                        "The barrier price separates them immediately"])
    _mk = dict(size=11, line=dict(width=1, color="rgba(0,0,0,0.5)"))
    for _m, _B in [(m, b) for m in MODES for b in BARRIERS]:
        _s = (dfB5.filter(((pl.col("mode") == _m) | (pl.col("amplitude") == 0))
                          & (pl.col("barrier") == _B))
                  .unique(subset=["amplitude"]).sort("amplitude"))
        fig.add_trace(go.Scatter(x=_s["vanilla_rmse_vp"].to_numpy(),
                                 y=_s["dV_vs_clean_bp"].to_numpy(),
                                 mode="markers+lines", name=f"{_m}, B/S0={_B}", marker=_mk,
                                 hovertemplate="RMSE %{x:.5f} vp<br>dV %{y:+.1f} bp"),
                      1, 1)
        fig.add_trace(go.Scatter(x=_s["repair_core_pct"].to_numpy(),
                                 y=_s["dV_vs_clean_pct"].to_numpy(),
                                 mode="markers+lines", name=f"{_m}, B/S0={_B}", marker=_mk,
                                 showlegend=False,
                                 hovertemplate="repair %{x:.2f}%<br>dV %{y:+.2f}%"),
                      1, 2)
    fig.add_hline(y=0, line_dash="dot", row=1, col=1)
    fig.add_hline(y=0, line_dash="dot", row=1, col=2)
    fig.update_xaxes(title_text="vanilla RMSE on the quoted lattice (vol pts)", row=1, col=1)
    fig.update_yaxes(title_text="barrier mispricing vs clean (bp of spot)", row=1, col=1)
    fig.update_xaxes(title_text="local-vol repair rate (% of vega-trusted grid)", row=1, col=2)
    fig.update_yaxes(title_text="barrier mispricing vs clean (% of premium)", row=1, col=2)
    fig.update_layout(width=1120, height=470,
                      title="B5 — the collocation blind spot priced: the horizontal axis on the "
                            "left barely moves, the vertical axis does not stop")
    fig.show()

## B6. Delta hedging — separating wrong Greeks from discrete-hedging noise

Mispricing is only half the cost. A desk that sells the barrier and hedges it with the wrong
delta bleeds P&L over the life of the trade even if it happened to sell at the right price. The
experiment isolates exactly that.

**Protocol.** Both desks sell the same option at the **same entry price** (the clean surface's
price), then hedge on the **same Heston truth paths** at the same $n$ monitoring dates, each
using its *own* delta interpolated from its *own* value function. Terminal P&L is

$$\Pi = V_{\text{entry}} + \sum_{j} \Delta_j\,(S_{j+1} - S_j)\;-\;\text{payoff},$$

with $\Delta_j$ forced to zero once the barrier has been breached, since the option is dead.

**Why the comparison is paired.** Discrete hedging of a barrier is intrinsically noisy: the
clean desk's own $\Pi$ has a standard deviation of a substantial fraction of the premium, driven
by gamma near the barrier and by the discreteness of the rebalancing. That noise is *common to
both desks on a given path*, so comparing two RMSEs computed independently mostly compares noise
— an early version of this experiment produced non-monotone differences for exactly that reason.
Taking the **per-path difference**

$$\delta\Pi \;=\; \sum_j \bigl(\Delta^{\text{dirty}}_j - \Delta^{\text{clean}}_j\bigr)(S_{j+1}-S_j)$$

cancels the common term exactly and leaves a pure "wrong Greeks" P&L whose distribution is
estimated to high precision from the same number of paths. Its standard deviation, its mean
absolute size, and its 5% tail are the model-risk numbers; the payoff and the entry price drop
out algebraically.

In [ ]:
def hedge_paired(desks, ref_name, kB, n_paths=None, seed=None, sub=None, chunk=None):
    """Both desks hedge the SAME Heston paths. Returns absolute stats for the reference desk
    and PAIRED difference stats for every other desk (common discrete-hedging noise cancels)."""
    n_paths = HEDGE_PATHS if n_paths is None else n_paths
    seed = SEED + 23 if seed is None else seed
    sub = MC_SUB if sub is None else sub
    chunk = CHUNK if chunk is None else chunk
    others = [m for m in desks if m != ref_name]
    acc = {m: dict(s=0.0, s2=0.0, sa=0.0, q=[]) for m in others}
    base = dict(s=0.0, s2=0.0)
    rng = np.random.default_rng(seed)
    n = 0
    entry = desks[ref_name]["price"]
    while n < n_paths:
        c = max(min(chunk, n_paths - n) // 2 * 2, 2)
        Xs = heston_chunk(T_EXP, N_MON, sub, c, rng)
        S = np.exp(Xs)
        alive = np.ones(c, bool)
        ah = np.ones((c, N_MON + 1), bool)
        for j in range(1, N_MON + 1):
            alive &= Xs[:, j] > kB
            ah[:, j] = alive
        payoff = np.where(ah[:, -1], np.maximum(S[:, -1] - np.exp(K_STRIKE), 0.0), 0.0)
        dS = S[:, 1:] - S[:, :-1]
        hedge_pnl = {}
        for m, d in desks.items():
            p = np.zeros(c)
            for j in range(N_MON):
                p += np.interp(Xs[:, j], d["x"], d["D"][j]) * ah[:, j] * dS[:, j]
            hedge_pnl[m] = p
        b = entry + hedge_pnl[ref_name] - payoff
        base["s"] += b.sum(); base["s2"] += (b ** 2).sum()
        for m in others:
            diff = hedge_pnl[m] - hedge_pnl[ref_name]
            acc[m]["s"] += diff.sum(); acc[m]["s2"] += (diff ** 2).sum()
            acc[m]["sa"] += np.abs(diff).sum(); acc[m]["q"].append(float(np.quantile(diff, 0.05)))
        n += c
    bm = base["s"] / n
    out = {ref_name: dict(pnl_mean=bm, pnl_rmse=float(np.sqrt(base["s2"] / n)),
                          pnl_sd=float(np.sqrt(max(base["s2"] / n - bm ** 2, 0))))}
    for m in others:
        mu = acc[m]["s"] / n
        sd = float(np.sqrt(max(acc[m]["s2"] / n - mu ** 2, 0)))
        out[m] = dict(paired_mean=mu, paired_sd=sd, paired_se=sd / np.sqrt(n),
                      paired_rmse=float(np.sqrt(acc[m]["s2"] / n)),
                      paired_mad=acc[m]["sa"] / n, paired_q05=float(np.mean(acc[m]["q"])))
    return out, n

In [ ]:
if RUN_SYNTHETIC and RUN_HEDGE:
    rows_B6 = []
    _t0 = time.time()
    for Bmn in BARRIERS:
        kB = np.log(Bmn)
        desks = {}
        for name, d in SURF.items():
            V, D, x, Vg, Dg = price_delta(d["vloc"], T_EXP, K_STRIKE, kB)
            desks[name] = dict(x=x, D=Dg, price=V, delta=D)
        res, n = hedge_paired(desks, "clean", kB)
        Vc = desks["clean"]["price"]
        for name in SURF:
            if name == "clean":
                rows_B6.append(dict(surface=name, mode=SURF[name]["mode"], amplitude=0.0,
                                    barrier=Bmn, n_paths=n,
                                    clean_pnl_mean=res["clean"]["pnl_mean"],
                                    clean_pnl_sd=res["clean"]["pnl_sd"],
                                    paired_sd=0.0, paired_mad=0.0, paired_q05=0.0,
                                    paired_sd_pct_premium=0.0, paired_q05_pct_premium=0.0,
                                    paired_sd_usd=0.0))
                continue
            r = res[name]
            rows_B6.append(dict(
                surface=name, mode=SURF[name]["mode"], amplitude=SURF[name]["a"],
                barrier=Bmn, n_paths=n,
                clean_pnl_mean=res["clean"]["pnl_mean"], clean_pnl_sd=res["clean"]["pnl_sd"],
                paired_sd=r["paired_sd"], paired_mad=r["paired_mad"], paired_q05=r["paired_q05"],
                paired_sd_pct_premium=100 * r["paired_sd"] / Vc,
                paired_q05_pct_premium=100 * r["paired_q05"] / Vc,
                paired_sd_usd=r["paired_sd"] * SPOT_REF * NOTIONAL))
    dfB6 = pl.DataFrame(rows_B6).sort(["mode", "barrier", "amplitude"])
    dfB6.write_parquet(OUT_DIR / "nb06_hedging.parquet")
    print(f"[B6] hedging done in {time.time()-_t0:.0f}s")
    print("\n[B6] model-risk hedging error: sd of the PAIRED P&L difference, "
          "as % of the clean premium")
    print(dfB6.pivot(values="paired_sd_pct_premium", index=["mode", "amplitude"], on="barrier")
              .sort(["mode", "amplitude"]))
    print("\n[B6] 5% tail of the paired difference (% of the clean premium) — "
          "the number a risk manager reads:")
    print(dfB6.pivot(values="paired_q05_pct_premium", index=["mode", "amplitude"], on="barrier")
              .sort(["mode", "amplitude"]))
    print("\n[B6] reference: the clean desk's OWN discrete-hedging P&L sd = "
          f"{dfB6['clean_pnl_sd'].max():.5f} (this is the common noise the pairing removes)")

In [ ]:
if RUN_SYNTHETIC and RUN_HEDGE:
    fig = go.Figure()
    for _m in MODES:
        for _B in BARRIERS:
            _s = (dfB6.filter(((pl.col("mode") == _m) | (pl.col("amplitude") == 0))
                              & (pl.col("barrier") == _B))
                      .unique(subset=["amplitude"]).sort("amplitude"))
            fig.add_trace(go.Scatter(x=_s["amplitude"].to_numpy(),
                                     y=_s["paired_sd_pct_premium"].to_numpy(),
                                     mode="markers+lines",
                                     name=f"{_m}, B/S0={_B}", marker=dict(size=10)))
    fig.update_layout(width=940, height=440,
                      title="B6 — model-risk hedging error from wrong Greeks "
                            "(paired, common hedging noise removed)",
                      xaxis_title="defect amplitude a (vanilla RMSE unchanged throughout)",
                      yaxis_title="sd of paired P&L difference (% of clean premium)")
    fig.show()

## B7. The digital — the butterfly defect read as a probability

A digital call is the derivative of the vanilla in strike,
$\mathbb{P}(S_T > K) = -\partial_K C$, so its price is a direct read of the risk-neutral
distribution function. That makes it the shortest bridge back to NB05 §C: where the butterfly
constraint fails, $g<0$, the density is negative, and the digital price is **non-monotone in
strike** — the surface asserts that a higher strike is more likely to finish in the money. This
is not a subtle pricing error; it is an arithmetic impossibility that a vanilla RMSE will
happily report as a good fit.

The digital is cheap to evaluate (no simulation: it is a closed-form functional of $w$ and
$\partial_k w$), and it is included as a second, terminal-payoff instrument so that the barrier's
message is not resting on a single product.

In [ ]:
def digital_call_clean(w_fn, k, tau, h=1e-5):
    """P(S_T > K) = N(d2) - phi(d2) * w_k / (2 sqrt(w)), the standard skew-adjusted digital."""
    k = np.atleast_1d(np.asarray(k, float))
    w = np.maximum(w_fn(k, tau), 1e-12)
    wk = (w_fn(k + h, tau) - w_fn(k - h, tau)) / (2 * h)
    sw = np.sqrt(w)
    d2 = -k / sw - sw / 2
    return _Phi(d2) - _phi(d2) * wk / (2 * sw)


if RUN_SYNTHETIC:
    _tau_d = float(np.exp(np.mean(np.log(QUOTE_TAUS))))
    _kk = np.linspace(KQ.min() * 0.95, KQ.max() * 0.95, 900)
    rows_B7 = []
    fig = go.Figure()
    for name, d in SURF.items():
        q = digital_call_clean(d["w_fn"], _kk, _tau_d)
        dq = np.diff(q)
        rows_B7.append(dict(surface=name, mode=d["mode"], amplitude=d["a"], tau=_tau_d,
                            vanilla_rmse_vp=d["rmse_vp"],
                            n_nonmonotone=int(np.sum(dq > 0)),
                            pct_nonmonotone=float(100 * np.mean(dq > 0)),
                            max_upward_step=float(dq.max()),
                            min_prob=float(q.min()), max_prob=float(q.max())))
        if name in ("clean", max(SURF, key=lambda n: SURF[n]["repair_core"])):
            fig.add_trace(go.Scatter(x=_kk, y=q, mode="lines", name=name))
    dfB7 = pl.DataFrame(rows_B7).sort(["mode", "amplitude"])
    dfB7.write_parquet(OUT_DIR / "nb06_digital.parquet")
    print(f"[B7] digital call at tau = {_tau_d:.2f}: a correct implied distribution function "
          f"is strictly DECREASING in k. `pct_nonmonotone` is the share of the strip where "
          f"the surface says otherwise.")
    print(dfB7)
    fig.update_layout(width=940, height=430,
                      title=f"B7 — implied P(S_T > K) at tau = {_tau_d:.2f}: "
                            f"the butterfly defect as a non-monotone distribution function",
                      xaxis_title="log-moneyness k", yaxis_title="implied P(S_T > K)")
    fig.show()

## B8. The same measurement on the real NB03/NB04 packs (gated)

The synthetic experiment gives a controlled dose and an independent truth, at the price of a
synthetic market. This section applies the identical barrier machinery to the actual exported
surfaces, where neither is available but the surfaces are the ones the thesis is about.

Two disciplines are enforced:

- **Fit parity.** A pair (clean, dirty) is only compared when their in-domain vanilla RMSE on
  the day's held-out quotes differ by less than `NB06_RMSE_TOL` vol points. Without this gate a
  price difference could simply be a fit-quality difference, and the comparison would say
  nothing about arbitrage. Pairs that fail the gate are reported as skipped, with their RMSEs,
  rather than silently dropped.
- **Domain discipline.** The barrier and the strike must sit strictly inside the pack's exported
  $(k,\tau)$ box, and the maturity is capped at the pack's own $\tau_{\max}$. Extrapolated local
  volatility is not the surface, and pricing a barrier on extrapolated wings would manufacture
  exactly the kind of artefact §B3 warns about.

No ground truth exists here, so the reported quantities are **differences between models**, not
errors: $\Delta V$, $\Delta$-delta, and each surface's repair rate. Read together with §B5 —
where the same differences can be calibrated against a known truth — they place the real
surfaces on the synthetic dose–response curve.

In [ ]:
def _scalar_text(x):
    a = np.asarray(x)
    x = a.reshape(-1)[0] if a.size == 1 else x
    if isinstance(x, (bytes, np.bytes_)):
        x = x.decode("utf-8", errors="replace")
    return str(x).strip()


def canonical_date(x):
    a = np.asarray(x)
    x = a.reshape(-1)[0] if a.size == 1 else x
    if isinstance(x, (bytes, np.bytes_)):
        x = x.decode("utf-8", errors="replace")
    if isinstance(x, np.datetime64):
        return np.datetime_as_string(x, unit="D")
    if isinstance(x, (_datetime, _date)):
        return x.strftime("%Y-%m-%d")
    s = str(x).strip().strip("\"'")
    if (s.startswith("b'") and s.endswith("'")) or (s.startswith('b"') and s.endswith('"')):
        s = s[2:-1]
    m = re.search(r"(\d{4})[-/]?(\d{2})[-/]?(\d{2})", s)
    if not m:
        raise ValueError(f"Unrecognized date value: {x!r}")
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

In [ ]:
def load_packs(folder):
    out = {}
    for fn in sorted(glob.glob(str(folder / "*.npz"))):
        z = np.load(fn, allow_pickle=True)
        model = _scalar_text(z["model"]); tag = _scalar_text(z["tag"])
        date = canonical_date(z["date"])
        key = (model if tag in ("", "lam10") else f"{model}_{tag}", date)
        out[key] = dict(k=np.asarray(z["k"], float), t=np.asarray(z["tau"], float),
                        W=np.asarray(z["w"], float), model=key[0], tag=tag, date=date)
    return out

In [ ]:
def load_day_quotes(date):
    if not REAL_PARQUET.exists():
        return None
    dk = canonical_date(date); compact = dk.replace("-", "")
    scan = pl.scan_parquet(REAL_PARQUET)
    cols = scan.collect_schema().names()
    iv_col = "iv_om" if "iv_om" in cols else "impl_volatility"
    if iv_col not in cols:
        return None
    want = [c for c in ["date", "tau", "k", iv_col] if c in cols]
    dtext = pl.col("date").cast(pl.Utf8)
    match = ((dtext.str.slice(0, 10) == dk) |
             (dtext.str.replace_all("-", "").str.slice(0, 8) == compact))
    otm = pl.col("is_otm") if "is_otm" in cols else pl.lit(True)
    df = (scan.filter(match & otm).select(want).drop_nulls([iv_col, "k", "tau"])
              .collect(engine="streaming"))
    return df.rename({iv_col: "iv"}) if df.height else None

In [ ]:
REAL_MAX_DAYS = int(os.environ.get("NB06_REAL_MAX_DAYS", "20"))
REAL_BARRIERS = [float(x) for x in os.environ.get("NB06_REAL_BARRIERS", "0.90,0.85").split(",")]

_packs = {}
if RUN_REAL and NB03_SURF.exists():
    _packs.update(load_packs(NB03_SURF))
if RUN_REAL and NB04_SURF.exists():
    _packs.update(load_packs(NB04_SURF))

if not RUN_REAL:
    print("[B8] gated off by NB06_SKIP_REAL=1.")
elif not _packs:
    print("[B8] gated: no NB03/NB04 surface packs found — run NB03/NB04 first, then re-run "
          "this notebook. Every other section is unaffected.")
else:
    _dates = sorted({d for (_, d) in _packs})
    if REAL_MAX_DAYS and len(_dates) > REAL_MAX_DAYS:
        _pick = np.linspace(0, len(_dates) - 1, REAL_MAX_DAYS).round().astype(int)
        _dates = [_dates[i] for i in sorted(set(_pick.tolist()))]
    print(f"[B8] {len(_packs)} pack(s); pricing on {len(_dates)} day(s); "
          f"models: {sorted({m for (m, _) in _packs})}")

In [ ]:
if RUN_REAL and _packs:
    rows_B8, skipped = [], []
    for date in _dates:
        day = {m: p for (m, d), p in _packs.items() if d == date}
        q = load_day_quotes(date)
        for m, p in sorted(day.items()):
            kg_, tg_, W_ = p["k"], p["t"], p["W"]
            T_d = float(min(T_EXP, tg_.max() * 0.98))
            if T_d < 0.15:
                skipped.append(dict(date=date, model=m, reason="pack tau range too short"))
                continue
            vfn, v, ia, ic, G, Wt, tr = dupire_fields(kg_, tg_, W_)
            rmse = float("nan")
            if q is not None:
                kq_, tq_, iv_ = q["k"].to_numpy(), q["tau"].to_numpy(), q["iv"].to_numpy()
                dom = ((kq_ >= kg_[0]) & (kq_ <= kg_[-1]) &
                       (tq_ >= tg_[0]) & (tq_ <= tg_[-1]))
                if dom.sum() >= 30:
                    itp = RegularGridInterpolator((tg_, kg_), W_, bounds_error=False,
                                                  fill_value=None)
                    w_m = np.maximum(itp(np.stack([tq_[dom], kq_[dom]], -1)), 1e-12)
                    rmse = float(np.sqrt(np.mean((np.sqrt(w_m / tq_[dom]) - iv_[dom]) ** 2)) * 100)
            for Bmn in REAL_BARRIERS:
                kB = float(np.log(Bmn))
                if kB <= kg_[0] + 0.02 or K_STRIKE >= kg_[-1] - 0.02:
                    skipped.append(dict(date=date, model=m,
                                        reason=f"barrier {Bmn} outside pack k-range"))
                    continue
                V, D, _, _, _ = price_delta(vfn, T_d, K_STRIKE, kB)
                rows_B8.append(dict(date=date, model=m, barrier=Bmn, maturity=T_d,
                                    price=V, delta=D, vanilla_rmse_vp=rmse,
                                    repair_core_pct=100 * float(ic.mean()),
                                    repair_all_pct=100 * float(ia.mean()),
                                    min_g=float(G[tr].min()) if tr.any() else float("nan")))
    if skipped:
        print("[B8] skips:")
        print(pl.DataFrame(skipped).group_by("reason").len().sort("len", descending=True))

In [ ]:
if RUN_REAL and _packs:
    if rows_B8:
        dfB8 = pl.DataFrame(rows_B8)
        dfB8.write_parquet(OUT_DIR / "nb06_real_barrier.parquet")
        print("\n[B8] per-model barrier price and repair rate (mean over days):")
        print(dfB8.group_by(["model", "barrier"]).agg(
            pl.col("price").mean().alias("mean_price"),
            pl.col("delta").mean().alias("mean_delta"),
            pl.col("vanilla_rmse_vp").mean().alias("mean_rmse_vp"),
            pl.col("repair_core_pct").mean().alias("mean_repair_pct"),
            pl.len().alias("n_days")).sort(["barrier", "model"]))

        # ---- fit-parity-gated clean/dirty pairs -----------------------------------------
        # Two pairing rules, both gated on fit parity:
        #   (a) the lambda=0 counterfactual: `<model>_lam0` is paired with `<model>`;
        #   (b) a global clean reference (NB06_CLEAN_REF), so NB04 families that have no
        #       lam0 twin -- the operators -- are still compared against a certified surface.
        CLEAN_REF = os.environ.get("NB06_CLEAN_REF", "")
        pairs = []
        for (date, Bmn), s in dfB8.group_by(["date", "barrier"], maintain_order=True):
            byname = {r["model"]: r for r in s.to_dicts()}
            todo = []
            for dirty in byname:
                if dirty.endswith("_lam0") and dirty[:-5] in byname:
                    todo.append((dirty[:-5], dirty, "lam0 counterfactual"))
            if CLEAN_REF and CLEAN_REF in byname:
                for dirty in byname:
                    if dirty != CLEAN_REF and not dirty.endswith("_lam0"):
                        todo.append((CLEAN_REF, dirty, "vs clean reference"))
            for clean, dirty, rule in todo:
                a, b = byname[clean], byname[dirty]
                gap = abs((a["vanilla_rmse_vp"] or np.nan) - (b["vanilla_rmse_vp"] or np.nan))
                pairs.append(dict(date=date, barrier=Bmn, rule=rule, clean=clean, dirty=dirty,
                                  rmse_clean=a["vanilla_rmse_vp"], rmse_dirty=b["vanilla_rmse_vp"],
                                  rmse_gap_vp=gap, fit_parity=bool(gap <= RMSE_MATCH_TOL_VP),
                                  repair_clean=a["repair_core_pct"],
                                  repair_dirty=b["repair_core_pct"],
                                  dV_bp=1e4 * (b["price"] - a["price"]),
                                  dV_pct=100 * (b["price"] / max(a["price"], 1e-12) - 1),
                                  ddelta=b["delta"] - a["delta"]))
        if not CLEAN_REF:
            print(f"\n[B8] note: set NB06_CLEAN_REF=<model> (e.g. 'ssvi', 'deep', "
                  f"'deeponet_prior') to also pair the operator families, which have no "
                  f"lam0 twin. Available: {sorted(dfB8['model'].unique().to_list())}")
        if pairs:
            dfP = pl.DataFrame(pairs)
            dfP.write_parquet(OUT_DIR / "nb06_real_pairs.parquet")
            ok = dfP.filter(pl.col("fit_parity"))
            print(f"\n[B8] clean/dirty pairs: {dfP.height} found, {ok.height} pass the "
                  f"fit-parity gate (|dRMSE| <= {RMSE_MATCH_TOL_VP} vp)")
            if dfP.height > ok.height:
                print("[B8] pairs FAILING the gate (their price gap is confounded by fit "
                      "quality and must not be read as an arbitrage effect):")
                print(dfP.filter(~pl.col("fit_parity"))
                         .group_by(["clean", "dirty"]).agg(
                             pl.col("rmse_gap_vp").mean().alias("mean_rmse_gap_vp"),
                             pl.len().alias("n")).sort("mean_rmse_gap_vp", descending=True))
            if ok.height:
                print(ok.group_by(["rule", "clean", "dirty", "barrier"]).agg(
                    pl.col("dV_bp").mean().alias("mean_dV_bp"),
                    pl.col("dV_pct").mean().alias("mean_dV_pct"),
                    pl.col("ddelta").mean().alias("mean_ddelta"),
                    pl.col("repair_dirty").mean().alias("mean_repair_dirty"),
                    pl.len().alias("n")).sort(["barrier", "dirty"]))
            else:
                print("[B8] no pair passes the gate — report the RMSE gaps rather than the "
                      "price differences, which would be confounded by fit quality.")
    else:
        print("[B8] no priceable model-days (see skips above).")

## Summary — what this notebook adds to the thesis

NB05 established that the arbitrage violations documented in NB03/NB04 are **not** a tradable
opportunity: there is essentially no executable static arbitrage in the quotes (§F1), and the
smoothing residual does not survive round-trip costs (§F2/§G). That answer is correct and it is
worth keeping, but on its own it invites the obvious question — if nobody can monetize the
defect, why does the defect matter?

This notebook answers it. The defect matters because the surface is not only a description of
the vanilla market; it is an input to the pricing and hedging of everything downstream. On a
path-dependent product:

- **§B3** — the same finite-difference audit that reports a violation can also *manufacture* one
  where the implied-vol inversion is ill-conditioned; a vega-trusted core separates the two, and
  the reference Heston surface reads exactly zero repair once it is applied.
- **§B4** — the numerical gap sits inside Monte Carlo noise and the Dupire round trip closes to a
  fraction of a basis point, so the pricer is validated; the local-vol-versus-Heston gap is
  quantified separately and is the scale against which everything else is judged.
- **§B5** — a defect that is *exactly invisible* to the vanilla objective moves the barrier price
  by an amount that grows without bound in the defect amplitude, while the vanilla RMSE column
  does not move in its fifth decimal.
- **§B6** — with the entry price held identical and the common discrete-hedging noise removed by
  pairing, the wrong deltas alone generate a model-risk P&L whose dispersion is a large fraction
  of the premium.
- **§B7** — the same defect makes the implied distribution function non-monotone: the surface
  asserts a negative probability.

The thesis sentence this supports:

> The value of enforcing no-arbitrage is not that it uncovers a free lunch — the vanilla market
> is efficient enough that it does not. It is that it prevents a surface which is
> indistinguishable from a certified one on the calibration objective from injecting
> economically material error into the price and the hedge of every path-dependent product
> derived from it.

**Three honest limitations**, each of which belongs in the text rather than in a footnote.
First, the controlled experiment uses a synthetic Heston market; the real-pack section (§B8)
lacks a ground truth and can therefore report differences but not errors. Second, the defect
geometry is imposed rather than learned — it is calibrated to reproduce a repair rate comparable
to the one measured on the operator, but it is a caricature of the operator's own failure mode.
Third, the local-volatility model is itself only one downstream use of the surface; a
stochastic-local-volatility desk would inherit the defect differently, and quantifying that is
outside the present scope.

**Run order.** NB01 → NB02 → NB03 → NB04 → NB05 → **NB06**. This notebook's synthetic sections
(§B1–B7) are self-contained and need no upstream artefact; only §B8 consumes the NB03/NB04
packs, and it skips cleanly when they are absent.